# 02 Bronze - Healthcare

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Preserves source values and lineage while converting each dataset to Delta.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; it is not an importable module.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'healthcare':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

specs = {'patients': {'filename': 'patients.csv', 'primary_key': ['patient_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'age_band', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'coverage_type', 'type': 'STRING', 'required': True}, {'name': 'risk_band', 'type': 'STRING', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'providers': {'filename': 'providers.csv', 'primary_key': ['provider_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'specialty', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'facility_type', 'type': 'STRING', 'required': True}, {'name': 'daily_capacity', 'type': 'BIGINT', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'appointments': {'filename': 'appointments.csv', 'primary_key': ['appointment_id'], 'foreign_keys': [['patient_id', 'patients', 'patient_id'], ['provider_id', 'providers', 'provider_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'appointment_id', 'type': 'STRING', 'required': True}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'booked_at', 'type': 'TIMESTAMP', 'required': True}, {'name': 'scheduled_start', 'type': 'TIMESTAMP', 'required': True}, {'name': 'scheduled_end', 'type': 'TIMESTAMP', 'required': True}, {'name': 'appointment_type', 'type': 'STRING', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'encounters': {'filename': 'encounters.csv', 'primary_key': ['encounter_id'], 'foreign_keys': [['appointment_id', 'appointments', 'appointment_id'], ['patient_id', 'patients', 'patient_id'], ['provider_id', 'providers', 'provider_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'encounter_id', 'type': 'STRING', 'required': True}, {'name': 'appointment_id', 'type': 'STRING', 'required': False}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'encounter_start', 'type': 'TIMESTAMP', 'required': True}, {'name': 'encounter_end', 'type': 'TIMESTAMP', 'required': True}, {'name': 'diagnosis_group', 'type': 'STRING', 'required': True}, {'name': 'procedure_group', 'type': 'STRING', 'required': True}, {'name': 'cost_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'disposition', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"appointments": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/appointments/", "encounters": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/encounters/", "patients": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/patients/", "providers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/providers/"}
destinations = {"appointments": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/appointments/", "encounters": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/encounters/", "patients": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/patients/", "providers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/providers/"}
landing_tables = {"appointments": f"{participant_key}_healthcare_appointments", "encounters": f"{participant_key}_healthcare_encounters", "patients": f"{participant_key}_healthcare_patients", "providers": f"{participant_key}_healthcare_providers"}

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_healthcare_patients (`participant_key` STRING, `source_row_id` STRING, `patient_id` STRING, `age_band` STRING, `region` STRING, `coverage_type` STRING, `risk_band` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/patients/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_healthcare_providers (`participant_key` STRING, `source_row_id` STRING, `provider_id` STRING, `specialty` STRING, `region` STRING, `facility_type` STRING, `daily_capacity` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/providers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_healthcare_appointments (`participant_key` STRING, `source_row_id` STRING, `appointment_id` STRING, `patient_id` STRING, `provider_id` STRING, `booked_at` STRING, `scheduled_start` STRING, `scheduled_end` STRING, `appointment_type` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/appointments/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_healthcare_encounters (`participant_key` STRING, `source_row_id` STRING, `encounter_id` STRING, `appointment_id` STRING, `patient_id` STRING, `provider_id` STRING, `encounter_start` STRING, `encounter_end` STRING, `diagnosis_group` STRING, `procedure_group` STRING, `cost_amount` STRING, `disposition` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/healthcare/encounters/'""")

for dataset, spec in specs.items():
    frame = (spark.table(f"aidp_lab.oci_landing.{landing_tables[dataset]}")
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp()))
    landing_count = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(destinations[dataset])
    bronze_count = spark.read.format("delta").load(destinations[dataset]).count()
    assert bronze_count == landing_count, f"Bronze count mismatch for {dataset}"
    print(f"Bronze {dataset}: {bronze_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_healthcare_patients (`participant_key` STRING, `source_row_id` STRING, `patient_id` STRING, `age_band` STRING, `region` STRING, `coverage_type` STRING, `risk_band` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/patients/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_healthcare_providers (`participant_key` STRING, `source_row_id` STRING, `provider_id` STRING, `specialty` STRING, `region` STRING, `facility_type` STRING, `daily_capacity` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/providers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_healthcare_appointments (`participant_key` STRING, `source_row_id` STRING, `appointment_id` STRING, `patient_id` STRING, `provider_id` STRING, `booked_at` STRING, `scheduled_start` STRING, `scheduled_end` STRING, `appointment_type` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/appointments/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_healthcare_encounters (`participant_key` STRING, `source_row_id` STRING, `encounter_id` STRING, `appointment_id` STRING, `patient_id` STRING, `provider_id` STRING, `encounter_start` STRING, `encounter_end` STRING, `diagnosis_group` STRING, `procedure_group` STRING, `cost_amount` STRING, `disposition` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/encounters/'""")


## Expected result

Four Landing CSV tables and four Bronze Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
